# 📏 Phi-3 Mini — Smart Unit Converter Tool
### Ask conversationally! e.g. *'convert 100 km to miles'* or *'how many pounds is 5 kg?'*
> ⚠️ **First:** Go to `Runtime → Change runtime type → T4 GPU` before running!

In [ ]:
# ✅ Cell 1 — Install Dependencies
!pip install transformers==4.41.2 accelerate -q
print('✅ All libraries installed!')

In [ ]:
# ✅ Cell 2 — Check GPU
!nvidia-smi
import torch
if torch.cuda.is_available():
    print(f'\n✅ GPU detected: {torch.cuda.get_device_name(0)}')
else:
    print('\n⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ✅ Cell 3 — All Imports
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import re
import textwrap

print('✅ All imports done!')
print(f'🖥️ Device: {"GPU - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (slow!)"}')

In [ ]:
# ✅ Cell 4 — Load Phi-3 Model
model_id = 'microsoft/Phi-3-mini-4k-instruct'

print('⏳ Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

print('⏳ Loading model... please wait (2-5 mins first time)')
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
    attn_implementation='eager'
)

pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
)

print('✅ Model loaded and ready!')

In [ ]:
# ✅ Cell 5 — Unit Conversion Engine (handles math precisely)

CONVERSION_TABLE = {
    # Length
    ('km', 'miles'): 0.621371,
    ('miles', 'km'): 1.60934,
    ('km', 'm'): 1000,
    ('m', 'km'): 0.001,
    ('m', 'cm'): 100,
    ('cm', 'm'): 0.01,
    ('m', 'ft'): 3.28084,
    ('ft', 'm'): 0.3048,
    ('ft', 'inches'): 12,
    ('inches', 'ft'): 0.0833333,
    ('cm', 'inches'): 0.393701,
    ('inches', 'cm'): 2.54,
    ('m', 'yards'): 1.09361,
    ('yards', 'm'): 0.9144,
    ('km', 'ft'): 3280.84,
    ('ft', 'km'): 0.0003048,
    ('miles', 'ft'): 5280,
    ('ft', 'miles'): 0.000189394,

    # Weight
    ('kg', 'lbs'): 2.20462,
    ('lbs', 'kg'): 0.453592,
    ('kg', 'grams'): 1000,
    ('grams', 'kg'): 0.001,
    ('kg', 'pounds'): 2.20462,
    ('pounds', 'kg'): 0.453592,
    ('grams', 'lbs'): 0.00220462,
    ('lbs', 'grams'): 453.592,
    ('grams', 'pounds'): 0.00220462,
    ('pounds', 'grams'): 453.592,
    ('kg', 'oz'): 35.274,
    ('oz', 'kg'): 0.0283495,
    ('lbs', 'oz'): 16,
    ('oz', 'lbs'): 0.0625,
    ('tonnes', 'kg'): 1000,
    ('kg', 'tonnes'): 0.001,

    # Speed
    ('kmh', 'mph'): 0.621371,
    ('mph', 'kmh'): 1.60934,
    ('km/h', 'mph'): 0.621371,
    ('mph', 'km/h'): 1.60934,
    ('ms', 'kmh'): 3.6,
    ('kmh', 'ms'): 0.277778,
    ('m/s', 'km/h'): 3.6,
    ('km/h', 'm/s'): 0.277778,
    ('knots', 'kmh'): 1.852,
    ('kmh', 'knots'): 0.539957,

    # Area
    ('sqm', 'sqft'): 10.7639,
    ('sqft', 'sqm'): 0.092903,
    ('sqkm', 'sqmiles'): 0.386102,
    ('sqmiles', 'sqkm'): 2.58999,
    ('hectares', 'acres'): 2.47105,
    ('acres', 'hectares'): 0.404686,

    # Volume
    ('liters', 'gallons'): 0.264172,
    ('gallons', 'liters'): 3.78541,
    ('liters', 'ml'): 1000,
    ('ml', 'liters'): 0.001,
    ('liters', 'pints'): 2.11338,
    ('pints', 'liters'): 0.473176,
    ('cups', 'ml'): 236.588,
    ('ml', 'cups'): 0.00422675,

    # Data
    ('gb', 'mb'): 1024,
    ('mb', 'gb'): 0.000976563,
    ('tb', 'gb'): 1024,
    ('gb', 'tb'): 0.000976563,
    ('mb', 'kb'): 1024,
    ('kb', 'mb'): 0.000976563,
    ('gb', 'kb'): 1048576,
    ('kb', 'gb'): 9.53674e-7,
}

def precise_convert(value: float, from_unit: str, to_unit: str):
    """Do precise unit conversion using lookup table."""
    from_unit = from_unit.lower().strip()
    to_unit = to_unit.lower().strip()

    # Temperature (special case)
    if from_unit in ['celsius', 'c'] and to_unit in ['fahrenheit', 'f']:
        return round((value * 9/5) + 32, 4)
    if from_unit in ['fahrenheit', 'f'] and to_unit in ['celsius', 'c']:
        return round((value - 32) * 5/9, 4)
    if from_unit in ['celsius', 'c'] and to_unit in ['kelvin', 'k']:
        return round(value + 273.15, 4)
    if from_unit in ['kelvin', 'k'] and to_unit in ['celsius', 'c']:
        return round(value - 273.15, 4)
    if from_unit in ['fahrenheit', 'f'] and to_unit in ['kelvin', 'k']:
        return round((value - 32) * 5/9 + 273.15, 4)
    if from_unit in ['kelvin', 'k'] and to_unit in ['fahrenheit', 'f']:
        return round((value - 273.15) * 9/5 + 32, 4)

    # Direct lookup
    if (from_unit, to_unit) in CONVERSION_TABLE:
        return round(value * CONVERSION_TABLE[(from_unit, to_unit)], 6)

    return None

print('✅ Conversion engine ready!')

In [ ]:
# ✅ Cell 6 — Phi-3 Understands Your Question + Gives Answer

def extract_conversion_info(user_query: str):
    """Use Phi-3 to extract: value, from_unit, to_unit from natural language."""

    system_prompt = """You are a unit conversion assistant. Extract the conversion details from the user's question.
Reply ONLY in this exact format with no extra text:
VALUE: <number>
FROM: <unit>
TO: <unit>

Examples:
User: convert 100 km to miles
VALUE: 100
FROM: km
TO: miles

User: how many pounds is 5 kg?
VALUE: 5
FROM: kg
TO: pounds

User: what is 37 celsius in fahrenheit?
VALUE: 37
FROM: celsius
TO: fahrenheit"""

    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_query}
    ]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    output = pipe(
        prompt,
        max_new_tokens=50,
        do_sample=False,
        temperature=None,
        top_p=None,
    )

    generated = output[0]['generated_text'][len(prompt):].strip()

    # Parse the structured output
    try:
        value_match = re.search(r'VALUE:\s*([\d.]+)', generated)
        from_match  = re.search(r'FROM:\s*(\S+)', generated)
        to_match    = re.search(r'TO:\s*(\S+)', generated)

        if value_match and from_match and to_match:
            return (
                float(value_match.group(1)),
                from_match.group(1).lower(),
                to_match.group(1).lower()
            )
    except:
        pass
    return None, None, None


def generate_explanation(user_query: str, value, from_unit, to_unit, result) -> str:
    """Use Phi-3 to give a natural language explanation of the result."""

    system_prompt = """You are a helpful unit converter assistant.
Give a short, clear, friendly answer with the conversion result.
Also add 1 interesting real-world context if relevant. Keep it under 4 sentences."""

    user_msg = f"""The user asked: '{user_query}'
The answer is: {value} {from_unit} = {result} {to_unit}
Explain this conversion result naturally."""

    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_msg}
    ]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    output = pipe(
        prompt,
        max_new_tokens=150,
        do_sample=False,
        temperature=None,
        top_p=None,
    )

    return output[0]['generated_text'][len(prompt):].strip()


def unit_converter_agent(user_query: str):
    print(f'\n📏 Query: {user_query}')
    print('🤔 Phi-3 is understanding your question...')

    value, from_unit, to_unit = extract_conversion_info(user_query)

    if value is None:
        print('❌ Could not understand the question.')
        return 'Sorry, I could not understand the conversion. Try: "convert 100 km to miles"'

    print(f'✅ Extracted → {value} {from_unit} to {to_unit}')
    print('🔢 Calculating precise result...')

    result = precise_convert(value, from_unit, to_unit)

    if result is None:
        return f'Sorry, I do not support converting {from_unit} to {to_unit} yet.'

    print(f'✅ Result: {result} {to_unit}')
    print('💬 Generating explanation...')

    explanation = generate_explanation(user_query, value, from_unit, to_unit, result)

    return f'\n📊 Result: {value} {from_unit} = {result} {to_unit}\n\n💬 {explanation}'

print('✅ Unit Converter Agent ready!')

In [ ]:
# ✅ Cell 7 — Start the Unit Converter!
print('=' * 60)
print('   📏 Phi-3 Mini — Smart Unit Converter')
print('=' * 60)
print('\n💡 You can ask like:')
print('   → convert 100 km to miles')
print('   → how many pounds is 5 kg?')
print('   → what is 37 celsius in fahrenheit?')
print('   → how many gallons is 10 liters?')
print('   → convert 1 TB to GB')
print('   → 60 mph to km/h')
print('\n   Type "quit" to exit')
print('=' * 60)

while True:
    user_input = input('\nYou: ').strip()
    if user_input.lower() in ['quit', 'exit', 'q']:
        print('👋 Goodbye!')
        break
    if not user_input:
        continue

    response = unit_converter_agent(user_input)
    print(f'\n🤖 Assistant:\n{textwrap.fill(response, width=80)}')